# <font color='gold'>Sampling Course (UWaterloo course) review text</font>


In [1]:
import pandas as pd
import numpy as np 
from nltk import word_tokenize
from sklearn.model_selection import StratifiedShuffleSplit
from sampling_util.table_util import display_count_table, compare_source_and_sample  # helper functions to make tables

#===========================================================
# Parameters
#===========================================================
random_state = 73

input_file = '../../data/1_processed/2_Uwaterloo_course_reviews.csv'

target_sample_size = 3000

rating_bins = None # use direct mapping for this file
rating_labels = {'[0.0]':'Neg', '[1.0]':'Pos','[NA]':'[NA]'}

length_bin_interval_size = 20

output_cols = ['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin']
#===========================================================

updated_sourcefile = input_file.replace('.csv', '_with_sampling_criteria.csv')
output_file_name = input_file.replace('1_processed/', '2_sampled/sampling_output_unlabelled/').replace('.csv', f'(N={target_sample_size}).csv')

rating_stats_file = input_file.replace('1_processed/', '2_sampled/sampling_output_stats/').replace('.csv', f'(N={target_sample_size})_rating_stats.csv')
length_stats_file = input_file.replace('1_processed/', '2_sampled/sampling_output_stats/').replace('.csv', f'(N={target_sample_size})_length_stats.csv')

print(f'\nInput:      {input_file}\n\n'
      f'Source_v2:  \033[032m{updated_sourcefile}\033[0m\n\n'
      f'Output:     \033[033m{output_file_name}\033[0m\n\n'
      f'Stats_1:    \033[35m{rating_stats_file}\033[0m\n'
      f'Stats_2:    \033[35m{length_stats_file}\033[0m\n')



Input:      ../../data/1_processed/2_Uwaterloo_course_reviews.csv

Source_v2:  ../../data/1_processed/2_Uwaterloo_course_reviews_with_sampling_criteria.csv

Output:     ../../data/2_sampled/sampling_output_unlabelled/2_Uwaterloo_course_reviews(N=3000).csv

Stats_1:    ../../data/2_sampled/sampling_output_stats/2_Uwaterloo_course_reviews(N=3000)_rating_stats.csv
Stats_2:    ../../data/2_sampled/sampling_output_stats/2_Uwaterloo_course_reviews(N=3000)_length_stats.csv



## <font color='cornflowerblue'>1) Prepare helper columns</font>


In [3]:
source_df = pd.read_csv(input_file)
print(source_df.shape)

# Get review length (as number of tokens)
source_df.loc[:, 'token_list'] = source_df.review_text.str.strip().apply(word_tokenize)
source_df.loc[:, 'num_token'] = source_df.token_list.apply(len)

# Get sampling_feature distribution
print("distinct courses: ", len(source_df.course_id.unique())) 
display_count_table(source_df, 'course_id')
display_count_table(source_df, 'num_token', sort_by='num_token')
display_count_table(source_df, 'rating', sort_by='rating', ascending=True)

print(f"\n\033[34m{'='*50}\033[0m\n") #===========================================

# Bin ratings into groups
source_df.loc[:, 'rating_bin'] = source_df.rating.copy().fillna('NA').apply(lambda x: f'[{x}]')
source_df.loc[:, 'rating_cat'] = source_df.rating_bin.map(rating_labels)  #or source_df.rating.apply(lambda x: 'Pos' if 4 < x <= 5 else 'Neg' if 1 <= x <= 2.5 else 'Neu/Conf')
display_count_table(source_df, ['rating_cat', 'rating_bin'], sort_by='rating_bin', ascending=True)
print()

# Bin num_token with the given interval size
interval_size = length_bin_interval_size
maxval = max(source_df.num_token.unique())
bin_edges = np.arange(1, maxval + interval_size, interval_size)

source_df.loc[:, 'length_bin'] = pd.cut(source_df.num_token, bins=bin_edges, labels=None, include_lowest=True)
display_count_table(source_df, 'length_bin', sort_by='length_bin', ascending=True)
print()

#=================================================================
# Create combined column for multi-criteria stratified sampling
source_df.loc[:, 'sample_criteria'] = source_df.rating_cat.astype(str) + '_' + source_df.length_bin.astype(str) 

display_count_table(source_df, 'sample_criteria', sort_by='sample_criteria')

source_df.to_csv(updated_sourcefile, index=False)
print(f'\nsource_df written to \033[33m{updated_sourcefile}\033[0m, done~!\n')
#=================================================================

source_df

(14810, 15)
distinct courses:  1730


,course_id,count
0,Algebra for Honours Mathematics,253
1,Introduction to Microeconomics,214
2,Calculus 1 for Honours Mathematics,171
3,Designing Functional Programs,146
4,Calculus 1 for Engineering,145
...,...,...
1725,Probability Models for Business and Accounting,1
1726,Canada: Cultures and Conflicts in the Colonial...,1
1727,Probability Theory and Statistics 2,1
1728,Economics and Life Cycle Cost Analysis,1


,num_token,count
55,1,102
29,2,199
19,3,229
7,4,258
21,5,227
...,...,...
280,581,1
255,588,1
257,590,1
266,635,1


,rating,count
1,0.0,4811
0,1.0,9778


,rating_cat,rating_bin,count
1,Neg,[0.0],4811
0,Pos,[1.0],9778
2,[NA],[NA],221


,length_bin,count
0,"(0.999, 21.0]",5214
1,"(21.0, 41.0]",3905
2,"(41.0, 61.0]",2265
3,"(61.0, 81.0]",1327
4,"(81.0, 101.0]",723
5,"(101.0, 121.0]",466
6,"(121.0, 141.0]",274
7,"(141.0, 161.0]",200
8,"(161.0, 181.0]",108
9,"(181.0, 201.0]",89


,sample_criteria,count
2,"Neg_(0.999, 21.0]",1730
12,"Neg_(101.0, 121.0]",157
14,"Neg_(121.0, 141.0]",99
16,"Neg_(141.0, 161.0]",70
21,"Neg_(161.0, 181.0]",41
...,...,...
43,"[NA]_(181.0, 201.0]",3
19,"[NA]_(21.0, 41.0]",54
20,"[NA]_(41.0, 61.0]",41
31,"[NA]_(61.0, 81.0]",9



source_df written to ../../data/1_processed/2_Uwaterloo_course_reviews_with_sampling_criteria.csv, done~!



,Unnamed: 0,id,entry_idx,course_code,course_id,num_ratings,useful,easy,liked,num_reviews,...,rating_category,rating,source,type,token_list,num_token,rating_bin,rating_cat,length_bin,sample_criteria
0,0,2_1_794,0,CS 115,Introduction to Computer Science 1,2111,21%,10%,23%,86,...,liked course,1.0,data2_14810,course review,"[go, to, office, hours, and, practice]",6,[1.0],Pos,"(0.999, 21.0]","Pos_(0.999, 21.0]"
1,1,2_2_794,1,CS 115,Introduction to Computer Science 1,2111,21%,10%,23%,86,...,disliked course,0.0,data2_14810,course review,"[One, of, my, least, favourite, courses, ., Al...",38,[0.0],Neg,"(21.0, 41.0]","Neg_(21.0, 41.0]"
2,2,2_3_794,2,CS 115,Introduction to Computer Science 1,2111,21%,10%,23%,86,...,disliked course,0.0,data2_14810,course review,"[It, starts, with, a, very, low, pace, but, af...",91,[0.0],Neg,"(81.0, 101.0]","Neg_(81.0, 101.0]"
3,3,2_4_794,3,CS 115,Introduction to Computer Science 1,2111,21%,10%,23%,86,...,liked course,1.0,data2_14810,course review,"[Took, this, in, 2018, with, no, programming, ...",57,[1.0],Pos,"(41.0, 61.0]","Pos_(41.0, 61.0]"
4,4,2_5_794,4,CS 115,Introduction to Computer Science 1,2111,21%,10%,23%,86,...,liked course,1.0,data2_14810,course review,"[I, loved, everything, about, cs, 115, ., Grea...",54,[1.0],Pos,"(41.0, 61.0]","Pos_(41.0, 61.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14805,14827,2_14806_63,14833,ASL 102R,American Sign Language 2,1,100%,75%,100%,1,...,liked course,1.0,data2_14810,course review,"[Took, this, online, with, Georgia, Whalen, .,...",127,[1.0],Pos,"(121.0, 141.0]","Pos_(121.0, 141.0]"
14806,14828,2_14807_1676,14834,CS 136L,Tools and Techniques for Software Development,1,100%,100%,100%,1,...,liked course,1.0,data2_14810,course review,"[I, either, designed, or, helped, design, the,...",38,[1.0],Pos,"(21.0, 41.0]","Pos_(21.0, 41.0]"
14807,14829,2_14808_1528,14835,AFM 417,Special Topics in Accounting,0,NaN,NaN,NaN,1,...,NaN,NaN,data2_14810,course review,"[Topic, 3, :, Intro, to, Data, Analytics]",7,[NA],[NA],"(0.999, 21.0]","[NA]_(0.999, 21.0]"
14808,14830,2_14809_1461,14836,ECON 472,Senior Honours Essay,0,100%,0%,NaN,1,...,NaN,NaN,data2_14810,course review,"[Not, sure, if, I, liked, this, course, yet, ....",74,[NA],[NA],"(61.0, 81.0]","[NA]_(61.0, 81.0]"


In [3]:
# ==================================================================================
# As sklearn requires bin with count > 1, 
# identify bins with fewer than 2 samples, duplicate that row (to be removed later)
# ==================================================================================
criteria_bin_counts = source_df['sample_criteria'].value_counts() 
big_bin_criteria_vals = criteria_bin_counts[criteria_bin_counts == max(criteria_bin_counts)].index[0]
single_count_bin_criteria_vals = criteria_bin_counts[criteria_bin_counts < 2].index
print("max_count_bin criteria vals:       ", big_bin_criteria_vals)
print("single_count_bin criteria vals::   ", single_count_bin_criteria_vals, '\n')

display_count_table(source_df, 'sample_criteria', sort_by='count', ascending=False) 
print(source_df.shape)

# For bins with only 1 sample, duplicate them so that the sampling module won't throw error. 
# Will drop duplicates after sampling if they exist in the selected samples
for criteria_value in single_count_bin_criteria_vals: 
    source_df = pd.concat([source_df, source_df.loc[source_df.sample_criteria == criteria_value]], ignore_index=True, sort=False)

display_count_table(source_df, 'sample_criteria', sort_by='count', ascending=False)
print(source_df.shape)

max_count_bin criteria vals:        Pos_(0.999, 21.0]
single_count_bin criteria vals::    Index(['Pos_(441.0, 461.0]', 'Pos_(421.0, 441.0]', '[NA]_(161.0, 181.0]',
       'Neg_(561.0, 581.0]', 'Pos_(621.0, 641.0]', 'Neg_(461.0, 481.0]',
       'Neg_(381.0, 401.0]', 'Neg_(841.0, 861.0]', 'Neg_(401.0, 421.0]'],
      dtype='object', name='sample_criteria') 



,sample_criteria,count
0,"Pos_(0.999, 21.0]",3394
1,"Pos_(21.0, 41.0]",2663
2,"Neg_(0.999, 21.0]",1730
3,"Pos_(41.0, 61.0]",1525
4,"Neg_(21.0, 41.0]",1188
...,...,...
52,"Pos_(621.0, 641.0]",1
55,"Pos_(421.0, 441.0]",1
54,"Pos_(441.0, 461.0]",1
53,"Neg_(841.0, 861.0]",1


(14810, 21)


,sample_criteria,count
0,"Pos_(0.999, 21.0]",3394
1,"Pos_(21.0, 41.0]",2663
2,"Neg_(0.999, 21.0]",1730
3,"Pos_(41.0, 61.0]",1525
4,"Neg_(21.0, 41.0]",1188
...,...,...
49,"[NA]_(161.0, 181.0]",2
48,"Neg_(401.0, 421.0]",2
47,"Neg_(381.0, 401.0]",2
46,"Neg_(581.0, 601.0]",2


(14819, 21)


## <font color='cornflowerblue'>2) Extract samples & post-processing</font>


In [4]:
# Get the stratified sample
sss = StratifiedShuffleSplit(n_splits=1, test_size=target_sample_size + len(single_count_bin_criteria_vals), random_state=random_state)
_train_idx, sample_idx = next(sss.split(source_df.copy(), source_df.sample_criteria))  # iterate through n_split, which is 1. Only need the 'sample_idx' variable
 
sample_df = source_df.iloc[sample_idx]
sample_df = sample_df[output_cols]

# now drop the duplicated row added earlier
print("before drop_duplicates:       ", sample_df.shape)
sample_df = sample_df.drop_duplicates(keep='first')
print("after  drop_duplicates:       ", sample_df.shape, '\n')

display_count_table(sample_df, 'sample_criteria', sort_by='count', ascending=False)
print()

# If current sample size > target_sample_size, adjust the sample size 
diff = len(sample_df) - target_sample_size
if diff > 0:  
    # diff > 0 means no duplicated rows were selected in this sample, so the drop_duplicates above made no change
    # In this case, drop same-number rows randomly from the largest sample_criteria category
    print(f"\033[35m\nNo duplicated rows in sample.\n\033[36mRemoving {diff} random-sample of sample_criteria == ({big_bin_criteria_vals}) to match target sample size !!!!!\n\033[0m")
    drop_scope = sample_df.loc[sample_df.sample_criteria == big_bin_criteria_vals]
    drop_row_idx = drop_scope.sample(n=diff).index
    sample_df = sample_df.drop(index=drop_row_idx)
    print(sample_df.shape, '\n')
    display_count_table(sample_df, 'sample_criteria', sort_by='count', ascending=False)
    print()
else:
    print(f"\033[36m\nDuplicated rows in sample already dropped, all good.\n\033[0m")
    
source_df = source_df.drop(columns=['token_list']).drop_duplicates(keep='first')

sample_df

before drop_duplicates:        (3009, 10)
after  drop_duplicates:        (3009, 10) 



,sample_criteria,count
0,"Pos_(0.999, 21.0]",689
1,"Pos_(21.0, 41.0]",541
2,"Neg_(0.999, 21.0]",351
3,"Pos_(41.0, 61.0]",310
4,"Neg_(21.0, 41.0]",241
5,"Pos_(61.0, 81.0]",182
6,"Neg_(41.0, 61.0]",142
7,"Pos_(81.0, 101.0]",92
8,"Neg_(61.0, 81.0]",86
9,"Pos_(101.0, 121.0]",61




No duplicated rows in sample.
Removing 9 random-sample of sample_criteria == (Pos_(0.999, 21.0]) to match target sample size !!!!!

(3000, 10) 



,sample_criteria,count
0,"Pos_(0.999, 21.0]",680
1,"Pos_(21.0, 41.0]",541
2,"Neg_(0.999, 21.0]",351
3,"Pos_(41.0, 61.0]",310
4,"Neg_(21.0, 41.0]",241
5,"Pos_(61.0, 81.0]",182
6,"Neg_(41.0, 61.0]",142
7,"Pos_(81.0, 101.0]",92
8,"Neg_(61.0, 81.0]",86
9,"Pos_(101.0, 121.0]",61


,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
11945,course review,data2_14810,2_11946_64,1.0,13,"Pos_(0.999, 21.0]","Fun course, boring at times though. Easy 90 fo...",[1.0],Pos,"(0.999, 21.0]"
8248,course review,data2_14810,2_8249_1120,1.0,26,"Pos_(21.0, 41.0]","Very fun course, it was enjoyable throughout. ...",[1.0],Pos,"(21.0, 41.0]"
6868,course review,data2_14810,2_6869_436,0.0,3,"Neg_(0.999, 21.0]",Bird mandatory course,[0.0],Neg,"(0.999, 21.0]"
11143,course review,data2_14810,2_11144_833,1.0,58,"Pos_(41.0, 61.0]",I LOVE graph theory so I loved this course. It...,[1.0],Pos,"(41.0, 61.0]"
5783,course review,data2_14810,2_5784_1024,1.0,9,"Pos_(0.999, 21.0]",if you like MATH135 you'll like this one,[1.0],Pos,"(0.999, 21.0]"
...,...,...,...,...,...,...,...,...,...,...
4470,course review,data2_14810,2_4471_234,1.0,104,"Pos_(101.0, 121.0]",Entire class was split into four tests worth 2...,[1.0],Pos,"(101.0, 121.0]"
1776,course review,data2_14810,2_1777_854,1.0,27,"Pos_(21.0, 41.0]",I took this online. It was a good course for t...,[1.0],Pos,"(21.0, 41.0]"
606,course review,data2_14810,2_607_164,1.0,153,"Pos_(141.0, 161.0]",I think the course material itself was somewha...,[1.0],Pos,"(141.0, 161.0]"
86,course review,data2_14810,2_87_54,1.0,5,"Pos_(0.999, 21.0]",Welcome to Waterloo Math.,[1.0],Pos,"(0.999, 21.0]"


## <font color='cornflowerblue'>3) Compare source vs. sample distributions</font>


In [5]:
rating_stats = compare_source_and_sample(source_df, sample_df, ['rating_cat', 'rating_bin'], sort_by='rating_bin', ascending=True).fillna('')
rating_stats.to_csv(rating_stats_file, index=False)
print(f'\nRating_stats written to \033[33m{rating_stats_file}\033[0m, done~!\n')

rating_stats


Rating_stats written to ../../data/2_sampled/sampling_output_stats/2_Uwaterloo_course_reviews(N=3000)_rating_stats.csv, done~!



,df_name,rating_cat,rating_bin,count,%,df_name,rating_cat,rating_bin,count,%
1,source_df,Neg,[0.0],4811.0,0.324848,sample_df,Neg,[0.0],979.0,0.326333
0,source_df,Pos,[1.0],9778.0,0.660230,sample_df,Pos,[1.0],1976.0,0.658667
2,source_df,[NA],[NA],221.0,0.014922,sample_df,[NA],[NA],45.0,0.015000
3,,,,14810.0,1.000000,,,,3000.0,1.000000


In [6]:
length_stats = compare_source_and_sample(source_df, sample_df, ['length_bin'], sort_by='length_bin', ascending=True).fillna('')
length_stats.to_csv(length_stats_file, index=False)
print(f'\nLength_stats written to \033[33m{length_stats_file}\033[0m, done~!\n')

length_stats


Length_stats written to ../../data/2_sampled/sampling_output_stats/2_Uwaterloo_course_reviews(N=3000)_length_stats.csv, done~!



,df_name,length_bin,count,%,df_name,length_bin,count,%
0,source_df,"(0.999, 21.0]",5214.0,0.352059,sample_df,"(0.999, 21.0]",1049.0,0.349667
1,source_df,"(21.0, 41.0]",3905.0,0.263673,sample_df,"(21.0, 41.0]",793.0,0.264333
2,source_df,"(41.0, 61.0]",2265.0,0.152937,sample_df,"(41.0, 61.0]",460.0,0.153333
3,source_df,"(61.0, 81.0]",1327.0,0.089602,sample_df,"(61.0, 81.0]",270.0,0.090000
4,source_df,"(81.0, 101.0]",723.0,0.048818,sample_df,"(81.0, 101.0]",147.0,0.049000
5,source_df,"(101.0, 121.0]",466.0,0.031465,sample_df,"(101.0, 121.0]",95.0,0.031667
6,source_df,"(121.0, 141.0]",274.0,0.018501,sample_df,"(121.0, 141.0]",55.0,0.018333
7,source_df,"(141.0, 161.0]",200.0,0.013504,sample_df,"(141.0, 161.0]",40.0,0.013333
8,source_df,"(161.0, 181.0]",108.0,0.007292,sample_df,"(161.0, 181.0]",21.0,0.007000
9,source_df,"(181.0, 201.0]",89.0,0.006009,sample_df,"(181.0, 201.0]",19.0,0.006333


## <font color='cornflowerblue'>4) Write to file</font>


In [7]:
print(sample_df.shape)
print(list(sample_df.columns.values), '\n')

sample_df.to_csv(output_file_name, index=False)
print(f'sample written to \033[33m{output_file_name}\033[0m, done~!\n')

sample_df

(3000, 10)
['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin'] 

sample written to ../../data/2_sampled/sampling_output_unlabelled/2_Uwaterloo_course_reviews(N=3000).csv, done~!



,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
11945,course review,data2_14810,2_11946_64,1.0,13,"Pos_(0.999, 21.0]","Fun course, boring at times though. Easy 90 fo...",[1.0],Pos,"(0.999, 21.0]"
8248,course review,data2_14810,2_8249_1120,1.0,26,"Pos_(21.0, 41.0]","Very fun course, it was enjoyable throughout. ...",[1.0],Pos,"(21.0, 41.0]"
6868,course review,data2_14810,2_6869_436,0.0,3,"Neg_(0.999, 21.0]",Bird mandatory course,[0.0],Neg,"(0.999, 21.0]"
11143,course review,data2_14810,2_11144_833,1.0,58,"Pos_(41.0, 61.0]",I LOVE graph theory so I loved this course. It...,[1.0],Pos,"(41.0, 61.0]"
5783,course review,data2_14810,2_5784_1024,1.0,9,"Pos_(0.999, 21.0]",if you like MATH135 you'll like this one,[1.0],Pos,"(0.999, 21.0]"
...,...,...,...,...,...,...,...,...,...,...
4470,course review,data2_14810,2_4471_234,1.0,104,"Pos_(101.0, 121.0]",Entire class was split into four tests worth 2...,[1.0],Pos,"(101.0, 121.0]"
1776,course review,data2_14810,2_1777_854,1.0,27,"Pos_(21.0, 41.0]",I took this online. It was a good course for t...,[1.0],Pos,"(21.0, 41.0]"
606,course review,data2_14810,2_607_164,1.0,153,"Pos_(141.0, 161.0]",I think the course material itself was somewha...,[1.0],Pos,"(141.0, 161.0]"
86,course review,data2_14810,2_87_54,1.0,5,"Pos_(0.999, 21.0]",Welcome to Waterloo Math.,[1.0],Pos,"(0.999, 21.0]"
